# FIXED COUPON BOND - QL EXAMPLE

This is a based on example in http://gouthamanbalaraman.com/blog/quantlib-bond-modeling.html

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from financepy.utils.date import Date
from financepy.utils.math import ONE_MILLION
from financepy.utils.day_count import DayCountTypes
from financepy.utils.frequency import FrequencyTypes
from financepy.utils.calendar import CalendarTypes, BusDayAdjustTypes, DateGenRuleTypes
from financepy.utils.schedule import Schedule

from financepy.market.curves.interpolator import InterpTypes
from financepy.market.curves.zero_rates_discount_curve import ZeroRatesDiscountCurve

from financepy.products.bonds.bond import Bond

#############################################################
#  FINANCEPY Version 1.1.2 - Built on 18 Sep 2026 at 20:26  #
#  This software is distributed FREE AND WITHOUT WARRANTY   #
#  Report issues at https://github.com/domokane/FinancePy   #
#############################################################



# Define the Bond

In [3]:
issue_dt = Date(15, 1, 2010)
maturity_dt = Date(15, 1, 2016)
coupon = 0.06
freq_type = FrequencyTypes.SEMI_ANNUAL
dc_type = DayCountTypes.THIRTY_360_BOND
face = ONE_MILLION

In [4]:
bond = Bond(issue_dt, maturity_dt, coupon, freq_type, dc_type)

In [5]:
print(bond)

OBJECT_TYPE: Bond
ISSUE DATE: 15-JAN-2010
MATURITY_DATE: 15-JAN-2016
COUPON (%): 6.0
FREQUENCY: FrequencyTypes.SEMI_ANNUAL
DAY_COUNT: DayCountTypes.THIRTY_360_BOND
EX-DIVIDEND DAYS: 0
CALENDAR TYPE: CalendarTypes.WEEKEND
BUS DAYS ADJUST: BusDayAdjustTypes.FOLLOWING
DATE GEN RULE: DateGenRuleTypes.BACKWARD
COUPON TYPE: CouponType.FIXED


To see the cash flows we first need to set the settlement date of the bond. 

In [6]:
settle_dt = Date(15, 1, 2015)

In [7]:
bond.print_payments(settle_dt, face)

Coupon Date 	 Payment Date 	         Status          Amount
15-JAN-2015 	             	     SETTLEMENT 
15-JUL-2015 	 15-JUL-2015 	      UNCHANGED 	        30000.00 
15-JAN-2016 	 15-JAN-2016 	       MATURITY 	      1030000.00 



## Discounting Bond Flows

We wish to define a zero rate curve. For this we need the dates and values of the zero rates.

In [8]:
zero_dts = [Date(15,1,2015), Date(15,7,2015), Date(15,1,2016)]
zero_rates = [0.00, 0.005, 0.007]

In [9]:
discount_curve = ZeroRatesDiscountCurve(settle_dt, zero_dts, zero_rates,
                                      FrequencyTypes.ANNUAL)

In [10]:
print(discount_curve)

OBJECT TYPE: ZeroRatesDiscountCurve
ZERO RATE FREQUENCY: FrequencyTypes.ANNUAL
DATES: ZERO RATES
15-JAN-2015:   0.00000000
15-JUL-2015:   0.00500000
15-JAN-2016:   0.00700000
ZERO RATE DC_TYPE: DayCountTypes.ACT_365F

OBJECT_TYPE: DiscountCurve
VALUE DATE: 15-JAN-2015
    DATES      TIMES(YRS) DISC FACTORS
 15-JAN-2015:     0.000000  1.00000000
 15-JUL-2015:     0.495890  0.99752978
 15-JAN-2016:     1.000000  0.99304866
INTERPOLATION TYPE: FLAT_FWD_RATES
TIME DAY COUNT TYPE: ACT_365F



In [11]:
discount_curve._times

array([0.        , 0.49589041, 1.        ])

In [12]:
discount_curve._dfs

array([1.        , 0.99752978, 0.99304866])

In [13]:
bond.print_payments(settle_dt)

Coupon Date 	 Payment Date 	         Status          Amount
15-JAN-2015 	             	     SETTLEMENT 
15-JUL-2015 	 15-JUL-2015 	      UNCHANGED 	            3.00 
15-JAN-2016 	 15-JAN-2016 	       MATURITY 	          103.00 



In [14]:
discount_curve.df(settle_dt)

1.0

In [15]:
bond.flow_amounts

[0.0, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03]

In [16]:
bond.accrued_interest(settle_dt, 1.0)

0.0

In [17]:
bond.clean_price_from_discount_curve(settle_dt, discount_curve)

105.27660126262175

In [18]:
bond.dirty_price_from_discount_curve(settle_dt, discount_curve)

105.27660126262175

As we are on the issue date of the bond there is a full coupon of accrued.

This agrees with QL which finds a clean price of 105.27654

Copyright (c) 2020 Dominic O'Kane